# F1 — Hands-On Lab: Fundamentals of ML, NLP & Deep Learning

**Foundation Track · Module F1 · 2 hours**

This notebook contains the three hands-on exercises that complete the F1 module:

| # | Lab | Duration | Tools |
|---|-----|----------|-------|
| 1 | Churn classification baseline — XGBoost + GridSearchCV + ROC-AUC + KS | ~35 min | scikit-learn, XGBoost, imbalanced-learn |
| 2 | Sentence embeddings on customer feedback — cluster & inspect themes | ~25 min | sentence-transformers (BGE-M3), scikit-learn |
| 3 | Model explainability — SHAP on the XGBoost baseline | ~20 min | SHAP, matplotlib |

> **How to use this notebook:** run cells top-to-bottom. Every lab is self-contained; nothing is downloaded from disk — datasets are generated synthetically so the lab is reproducible offline. Each lab ends with a **"What to look for"** discussion block — that is the case-analysis material trainees will discuss with the trainer.

## 0. Environment setup

Run this once at the start of the session. If a package is already installed it'll be a no-op.

In [ ]:
#%pip install -q -U numpy pandas matplotlib scikit-learn xgboost shap

In [ ]:
# One-time install — uncomment the line below if any package is missing.
# This lab needs only: scikit-learn, xgboost, shap, matplotlib, pandas, numpy.
# (sentence-transformers is optional — Lab 2 falls back to TF-IDF automatically.)
# !pip install -q -U scikit-learn xgboost shap matplotlib pandas numpy

# --- Clean, quiet environment -------------------------------------------------
import os, warnings, logging


warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"


for _name in ("matplotlib", "shap", "numexpr", "xgboost"):
    logging.getLogger(_name).setLevel(logging.ERROR)

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg") if os.environ.get("MPLBACKEND") == "Agg" else None
import matplotlib.pyplot as plt

np.random.seed(42)
print("Environment ready.")
print("NumPy:", np.__version__, "| Pandas:", pd.__version__)

---
# Lab 1 — Churn classification baseline with XGBoost

**Goal.** Build a churn classifier on a tabular dataset, walk every step of a *real* classical-ML pipeline, then tune it with `GridSearchCV` and report two industry metrics: **ROC-AUC** and the **Kolmogorov–Smirnov (KS) statistic**.

**Why this matters.** This is the workhorse of every BFSI / telecom / SaaS retention team. The same recipe — encode → scale → handle imbalance → tune → measure — is what you'll wire into production.

### Step 1.1 — Generate a realistic telco-style churn dataset

We'll synthesise a 5,000-row dataset with realistic signal so the lab runs anywhere, no internet needed.

In [ ]:
# Synthesise a telco-churn-style dataset with mixed types, missingness, and class imbalance
from sklearn.datasets import make_classification

N = 5000
X_raw, y = make_classification(
    n_samples=N, n_features=12, n_informative=7, n_redundant=2,
    n_classes=2, weights=[0.85, 0.15],   # ~15% churn — realistic imbalance
    flip_y=0.03, class_sep=1.1, random_state=42,
)

feature_names = [
    "tenure_months", "monthly_charges", "total_charges", "num_support_calls",
    "data_usage_gb", "avg_session_min", "late_payments_12m", "num_services",
    "discount_pct", "complaints_90d", "nps_score", "device_age_months",
]
df = pd.DataFrame(X_raw, columns=feature_names)

# Bring values to a human scale
df["tenure_months"]      = (df["tenure_months"] * 12 + 36).clip(1, 72).round()
df["monthly_charges"]    = (df["monthly_charges"] * 25 + 75).clip(20, 250).round(2)
df["total_charges"]      = (df["monthly_charges"] * df["tenure_months"]).round(2)
df["num_support_calls"]  = (df["num_support_calls"] * 2 + 3).clip(0, 25).round()
df["data_usage_gb"]      = (df["data_usage_gb"] * 20 + 50).clip(0, 500).round(1)
df["avg_session_min"]    = (df["avg_session_min"] * 5 + 12).clip(0, 90).round(1)
df["late_payments_12m"]  = (df["late_payments_12m"] + 2).clip(0, 12).round()
df["num_services"]       = (df["num_services"] + 4).clip(1, 9).round()
df["discount_pct"]       = (df["discount_pct"] * 5 + 10).clip(0, 40).round(1)
df["complaints_90d"]     = (df["complaints_90d"] + 1).clip(0, 8).round()
df["nps_score"]          = (df["nps_score"] * 2 + 6).clip(0, 10).round()
df["device_age_months"]  = (df["device_age_months"] * 6 + 18).clip(0, 60).round()

# Inject categoricals — these need encoding
df["contract_type"] = np.random.choice(
    ["Month-to-month", "One year", "Two year"], size=N, p=[0.55, 0.25, 0.20])
df["payment_method"] = np.random.choice(
    ["Auto-Bank", "Auto-Card", "Mailed-Check", "Electronic-Check"], size=N, p=[0.30, 0.30, 0.15, 0.25])

# Inject ~5% missingness in two columns — production data is never clean
miss_idx = np.random.choice(N, size=int(0.05 * N), replace=False)
df.loc[miss_idx, "total_charges"] = np.nan
miss_idx2 = np.random.choice(N, size=int(0.03 * N), replace=False)
df.loc[miss_idx2, "nps_score"] = np.nan

df["churn"] = y
print("Shape:", df.shape, "| Churn rate:", round(df['churn'].mean(), 3))
df.head()

In [ ]:
# Quick health check — class imbalance, dtypes, nulls


### Step 1.2 — Feature engineering pipeline

We build *one* `ColumnTransformer` that does everything before modelling:

- **Numeric:** median imputation → standard scaling.
- **Categorical:** most-frequent imputation → one-hot encoding.

Keep the imputation/encoding *inside* the pipeline — never fit them on the full data before split, or you'll leak the test distribution into training.

### Step 1.3 — Handle class imbalance with `scale_pos_weight`

Only ~15% of customers churn. A naïve classifier will happily predict "no churn" for
everyone and report ~85% accuracy while being useless.

We correct for this with **XGBoost's built-in `scale_pos_weight`**, which up-weights the
minority (churn) class directly inside the training loss. Set it to
`n_negative / n_positive` and the model pays proportionally more attention to the rare
class — the same goal as resampling, with **no extra dependency** and nothing that can
break on a library-version mismatch.

> **Why not SMOTE here?** SMOTE (synthetic over-sampling) is the other common fix, and we
> show it as an *optional* cell in Step 1.3b. In production, `scale_pos_weight` (or
> `class_weight`) is usually the cleaner first choice for tree models: fewer moving parts,
> no synthetic data, and identical handling at train and inference time. Reach for SMOTE
> when class weighting alone isn't enough and the minority class is sparsely covered in
> feature space.

### Step 1.3b — (Optional) the SMOTE alternative

This cell is **optional** and **safe to skip**. It demonstrates the resampling approach for
teaching completeness. It only runs if `imbalanced-learn` is importable; otherwise it prints
a note and does nothing — it can never raise, so the lab always continues cleanly.

In [ ]:
# Optional: SMOTE-based pipeline, shown for comparison. Guarded so it never errors.


### Step 1.4 — Hyperparameter tuning with `GridSearchCV`

We tune a small grid on three high-leverage knobs: tree depth, learning rate, and number of trees. Scoring is `roc_auc` because that's what the business will look at.

> 💡 **For production:** swap `GridSearchCV` for **Optuna** (Bayesian search) once the grid grows past ~30 combinations. Code shown at the bottom of this section for reference.

### Step 1.5 — Evaluate: ROC-AUC and the KS statistic

**ROC-AUC** — probability that a random positive is ranked higher than a random negative. 0.5 = random, 1.0 = perfect. **>0.75 is usually production-grade for churn.**

**KS (Kolmogorov–Smirnov) statistic** — the *maximum vertical distance* between the cumulative distribution of predicted scores for positives vs negatives. This is what risk and credit teams live by. **>0.30 is decent; >0.50 is strong.** Where KS peaks is also where you'd set your decision threshold.

In [ ]:
from sklearn.metrics import roc_auc_score, classification_report, roc_curve



# KS statistic — max separation between class-1 and class-0 score CDFs
def ks_statistic(y_true, y_score):
    df_ks = pd.DataFrame({'y': y_true, 's': y_score}).sort_values('s', ascending=False).reset_index(drop=True)
    df_ks['cum_pos'] = (df_ks['y'] == 1).cumsum() / max((df_ks['y'] == 1).sum(), 1)
    df_ks['cum_neg'] = (df_ks['y'] == 0).cumsum() / max((df_ks['y'] == 0).sum(), 1)
    df_ks['gap'] = (df_ks['cum_pos'] - df_ks['cum_neg']).abs()
    ks_value = df_ks['gap'].max()
    ks_threshold = df_ks.loc[df_ks['gap'].idxmax(), 's']
    return ks_value, ks_threshold, df_ks

ks, ks_thr, df_ks = ks_statistic(y_test, y_proba)
print(f"ROC-AUC : {auc:.4f}")
print(f"KS      : {ks:.4f}  (max separation at score ≈ {ks_thr:.3f})")
print("\nClassification report (threshold = 0.5):\n")
print(classification_report(y_test, y_pred, target_names=['No-churn', 'Churn']))

In [ ]:
# Visualise ROC and KS side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[0].plot(fpr, tpr, lw=2.2, label=f'XGBoost (AUC={auc:.3f})')
axes[0].plot([0, 1], [0, 1], '--', color='gray', alpha=0.6, label='Random')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

xs = np.arange(len(df_ks)) / len(df_ks)
axes[1].plot(xs, df_ks['cum_pos'], lw=2, label='Cumulative % churners')
axes[1].plot(xs, df_ks['cum_neg'], lw=2, label='Cumulative % non-churners')
ks_x = df_ks['gap'].idxmax() / len(df_ks)
axes[1].vlines(ks_x, df_ks.loc[df_ks['gap'].idxmax(), 'cum_neg'],
               df_ks.loc[df_ks['gap'].idxmax(), 'cum_pos'],
               linestyles='--', color='red', label=f'KS = {ks:.3f}')
axes[1].set_xlabel('Population (sorted by score, descending)')
axes[1].set_ylabel('Cumulative %'); axes[1].set_title('KS Plot')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 🔍 Lab 1 — What to look for (case discussion)

Discuss with the cohort:

1. **Why is accuracy a dishonest metric here?** Predicting "no-churn" for everyone gives ~85% accuracy and zero business value.
2. **Where does the KS curve peak?** That's the natural decision threshold for the *retention-cost-vs-acquisition-cost* trade-off — almost never 0.5.
3. **SMOTE inside or outside the pipeline?** It belongs *inside*, *after* the train/CV split. If you SMOTE the whole dataset and then split, you've contaminated the test set with synthetic neighbours of test rows.
4. **What single hyperparameter usually matters most for XGBoost on tabular data?** Tree depth + learning rate together — they trade off. With `max_depth=3` and `lr=0.05` you need more trees; with `max_depth=6` you risk overfit.
5. **When would you reach for Optuna over GridSearchCV?** When the grid > ~30 cells or you want to tune continuous params (`lr`, `subsample`) without arbitrary discretisation.

> **Optional reference snippet** — same tuning with Optuna (commented out for time):

```python
# import optuna
# def objective(trial):
#     params = {
#         'clf__n_estimators':  trial.suggest_int('n_est', 100, 400),
#         'clf__max_depth':     trial.suggest_int('depth', 3, 7),
#         'clf__learning_rate': trial.suggest_float('lr', 0.01, 0.2, log=True),
#         'clf__subsample':     trial.suggest_float('sub', 0.6, 1.0),
#     }
#     full_pipe.set_params(**params)
#     from sklearn.model_selection import cross_val_score
#     return cross_val_score(full_pipe, X_train, y_train, cv=cv, scoring='roc_auc', n_jobs=-1).mean()
# study = optuna.create_study(direction='maximize')
# study.optimize(objective, n_trials=30)
# print(study.best_params, study.best_value)
```

---
# Lab 2 — Sentence embeddings on customer feedback (BGE-M3)

**Goal.** Turn short customer feedback into dense vectors with **BGE-M3**, cluster the vectors, and inspect what themes emerge — *without* labelling anything.

**Why this matters.** This is the simplest possible NLP-and-deep-learning workflow that delivers real value to a business: a 1-hour analyst pipeline can replace a 2-week qualitative coding project.

### Step 2.1 — Build a small but realistic feedback corpus

In [ ]:
feedback_corpus = [
    # Network / signal complaints
    "Network drops constantly in my area, voice calls cut off every few minutes",
    "Signal strength is awful indoors, I can't even make calls from my bedroom",
    "Internet has been down for 3 days, no resolution from support yet",
    "5G coverage is non-existent in my locality despite promises in the ad",
    "Calls keep dropping when I move between rooms, very frustrating",
    "Mobile data speed is painfully slow during peak hours",
    "Network reliability is terrible after the latest tower upgrade",
    "Wi-Fi router keeps disconnecting every couple of hours",

    # Billing / pricing complaints
    "I was charged twice for the same month, please refund immediately",
    "Hidden charges keep appearing on my bill every month",
    "The price hike was not communicated to me at all",
    "My plan was changed without consent and I'm being overcharged",
    "Auto-debit took the wrong amount from my account, this is unacceptable",
    "Bill went up 40% with no warning and no clear explanation",
    "Discount code from the website was never applied to my invoice",

    # Support / service experience
    "Customer service agent was rude and unhelpful, took 45 minutes",
    "Chatbot kept looping the same answer and I never reached a human",
    "Support ticket from last week is still unresolved, no follow-up",
    "Agent transferred me four times before disconnecting the call",
    "Waited on hold for over an hour just to confirm a payment",
    "The technician never showed up for the scheduled installation",
    "Filed a complaint two weeks ago and got zero response",

    # Praise / positive
    "Excellent service, the support team resolved my issue in minutes",
    "Very happy with the new fibre plan, speeds are exactly as advertised",
    "The new app is so much easier to use than the old one",
    "Quick installation and the technician was very professional",
    "Best customer support experience I've had in years, well done",
    "Loving the unlimited data plan, no throttling so far",

    # App / UX
    "App crashes every time I try to view my bill",
    "Can't log in to the customer portal since the latest update",
    "The mobile app is buggy and slow, payments fail half the time",
    "Auto-pay setting keeps resetting itself in the app",
]
print(f"Corpus size: {len(feedback_corpus)} feedback items")

### Step 2.2 — Generate embeddings with BGE-M3

`BGE-M3` (BAAI General Embeddings, multi-lingual / multi-functionality / multi-granularity) is a strong open-source embedding model — multilingual, supports up to 8192 tokens, free of cost. If your training laptop can't download the model, the fallback cell below uses scikit-learn's TF-IDF so the rest of the lab still works.

In [ ]:
# Primary path — BGE-M3 via sentence-transformers.
# Falls back to TF-IDF + SVD if the model isn't installed, so the lab always runs.
USING_BGE = False
try:
    from sentence_transformers import SentenceTransformer
    print("Loading BGE-M3 (downloads ~2.3 GB on first run)...")
    model = SentenceTransformer("BAAI/bge-m3")
    embeddings = model.encode(feedback_corpus, normalize_embeddings=True,
                              show_progress_bar=False)
    USING_BGE = True
    print(f"BGE-M3 embeddings ready. Shape: {embeddings.shape}")
except Exception as exc:
    print(f"BGE-M3 not available ({type(exc).__name__}) — using a TF-IDF + SVD fallback.")
    print("The clustering workflow is identical; only the vector source changes.")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.preprocessing import normalize

    tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_df=0.95,
                            stop_words="english")
    X_tfidf = tfidf.fit_transform(feedback_corpus)

    # Reduce the sparse TF-IDF space to a few dense dimensions so distances are
    # meaningful on a small corpus (raw high-dim sparse vectors cluster poorly).
    n_comp = min(8, X_tfidf.shape[1] - 1, len(feedback_corpus) - 1)
    svd = TruncatedSVD(n_components=n_comp, random_state=42)
    embeddings = normalize(svd.fit_transform(X_tfidf))
    print(f"TF-IDF + SVD embeddings ready. Shape: {embeddings.shape}")

### Step 2.3 — Cluster the embeddings with K-Means

We try `k = 4` first because we *suspect* there are roughly four themes (network / billing / support / praise) — in production you'd run the elbow / silhouette method to pick `k`.

### Step 2.4 — Visualise the clusters in 2-D

We project the high-dimensional embeddings to 2-D with t-SNE so we can eyeball the cluster structure.

In [ ]:
from sklearn.manifold import TSNE

perp = min(15, max(2, len(feedback_corpus) // 3))   # safe perplexity for tiny corpora
xy = TSNE(n_components=2, perplexity=perp, random_state=42, init='random').fit_transform(embeddings)

plt.figure(figsize=(9, 6))
palette = plt.cm.Set2(np.linspace(0, 1, K))
for c in range(K):
    mask = labels == c
    plt.scatter(xy[mask, 0], xy[mask, 1], s=120, color=palette[c],
                edgecolors='black', linewidths=0.6, label=f'Cluster {c}')
plt.title(f"Customer feedback clusters ({'BGE-M3' if USING_BGE else 'TF-IDF'} embeddings → t-SNE)")
plt.xlabel('t-SNE dim 1'); plt.ylabel('t-SNE dim 2')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Lab 2 — What to look for (case discussion)

Discuss with the cohort:

1. **Did the clusters match the four themes we baked in?** If a billing complaint landed in the support cluster, *why*? Usually because the vocabulary overlapped (`"agent never refunded me"` has both billing and support keywords).
2. **Why is `normalize_embeddings=True` important?** Cosine similarity assumes unit-length vectors. With normalisation, K-Means' Euclidean distance becomes monotonic with cosine distance.
3. **BGE-M3 vs. TF-IDF — when does the gap matter?** On longer, semantically-richer text. For 10–20 word complaint snippets TF-IDF can be surprisingly competitive — *use the embedding model that earns its cost on your data, not on a benchmark.*
4. **When does this fall over?** Out-of-domain language, sarcasm, code-mixed text. BGE-M3 handles multilingual well; sarcasm needs a classifier.
5. **What would you build next?** A small classifier on top of the embeddings (logistic regression on the vectors) — supervised, ~50 labels, production-ready in a day.

---
# Lab 3 — Explainability with SHAP on the XGBoost baseline

**Goal.** Open the XGBoost model from Lab 1, compute **SHAP** values, plot global feature importance, and translate the top drivers into a sentence a business stakeholder can act on.

**Why this matters.** A model that nobody trusts gets unplugged. SHAP turns "black box" into "here's exactly which features pushed *this customer* over the churn threshold, and by how much."

### Step 3.1 — Extract the trained XGBoost from Lab 1's pipeline

### Step 3.2 — Compute SHAP values

For tree models, `shap.TreeExplainer` is *exact* and fast — no sampling, no approximation. It returns the contribution of each feature to each prediction, measured as the deviation from the model's average prediction.

In [ ]:
explainer   = shap.TreeExplainer(fitted_xgb)
shap_values = explainer.shap_values(X_test_proc_df)
print(f"SHAP values shape: {shap_values.shape}  (rows × features)")
print(f"Expected value (base log-odds prediction): {explainer.expected_value:.4f}")

### Step 3.3 — Global feature importance (summary plot)

Each dot is one customer. **X-position** is the SHAP value (how much that feature shifted the prediction). **Colour** is the feature's raw value (red high, blue low). Features are ordered top-down by mean absolute impact.

### Step 3.4 — Top drivers, in business language

We extract the top-5 features by mean absolute SHAP value and translate each into a plain-English narrative the retention team can use in talking points.

In [ ]:
# Bar plot of top drivers
plt.figure(figsize=(8.5, 4.5))
plt.barh(top_drivers['feature'][::-1], top_drivers['mean_abs_shap'][::-1], color='#0D7C66')
plt.xlabel('Mean |SHAP value|  (avg. impact on log-odds of churn)')
plt.title('Top global drivers of the churn model')
plt.grid(alpha=0.3, axis='x'); plt.tight_layout(); plt.show()

### Step 3.5 — One-customer waterfall (local explanation)

Globals tell you *which features matter*. Locals tell you *why this particular customer was flagged*. The waterfall plot below decomposes a single prediction into feature contributions stacked on top of the model's base value.

In [ ]:
# Pick a high-risk test customer (predicted probability of churn > 0.7) for an illustrative explanation
high_risk = np.where(y_proba > 0.7)[0]
i = high_risk[0] if len(high_risk) else 0

print(f"Customer index : {i}")
print(f"Predicted P(churn) : {y_proba[i]:.3f}")
print(f"Actual label       : {'CHURN' if y_test[i] == 1 else 'NO-CHURN'}\n")

shap_exp = shap.Explanation(
    values=shap_values[i],
    base_values=explainer.expected_value,
    data=X_test_proc_df.iloc[i].values,
    feature_names=all_feature_names,
)
shap.plots.waterfall(shap_exp, max_display=10, show=False)
plt.tight_layout(); plt.show()

### Lab 3 — What to look for (case discussion)

Discuss with the cohort:

1. **Which features showed up at the top?** Across most synthetic runs, `complaints_90d`, `late_payments_12m`, `num_support_calls` and `contract_type_Month-to-month` dominate — match your business intuition?
2. **SHAP vs. plain feature importance from XGBoost** — why prefer SHAP?
   - SHAP shows *direction* (red = high value pushes towards churn; blue = low value).
   - SHAP works at the *individual customer* level, not just globally.
   - SHAP is theoretically grounded in cooperative game theory (Shapley values) — feature attributions are guaranteed to sum to the prediction.
3. **What if a "logical" driver is missing from the top-5?** That's the model telling you it has *better* signal elsewhere. Don't override it on gut — test whether dropping the feature hurts AUC.
4. **SHAP vs LIME — when does LIME still earn its place?**
   - LIME is **model-agnostic** (text, image, tabular) and **per-instance fast**.
   - SHAP gives **global + local** consistency and exact values for trees.
   - **Rule of thumb:** SHAP for tabular + tree models; LIME for text / image black-box models where TreeSHAP doesn't apply.
5. **How does this connect to the EU AI Act?** Article 13 (transparency) and Article 14 (human oversight) of the EU AI Act effectively require this kind of *meaningful explanation* for any high-risk decisioning system — which churn-driven retention offers can fall under, depending on jurisdiction. SHAP is the closest thing the industry has today to a default answer.

---
# ✅ F1 Lab — Wrap-up checklist

Before you close the notebook, confirm you can answer each of these from memory:

- [ ] What's the difference between *supervised*, *unsupervised*, and *self-supervised* learning?
- [ ] In a class-imbalanced problem, which two metrics did we report and why is accuracy alone a trap?
- [ ] Where in the pipeline does SMOTE belong — and why never *before* the split?
- [ ] Why do embeddings + clustering give you a "free" first-pass at unstructured feedback?
- [ ] What's the difference between *global* and *local* SHAP explanations?
- [ ] When would you reach for LIME over SHAP?
- [ ] What are PSI and CSI and why do they matter in production?  *(covered in the deck — not the lab)*

Bring any unanswered ones to the trainer for the assessment debrief.

> **Next session — F2:** AI, Agentic AI, LLMs & Prompt & Context Engineering.